## 1. Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import Dense, Flatten, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 2. Load Tabular Dataset

In [2]:
df = pd.read_csv("data/tabular.csv")

print(df.head())

        image  bedrooms  bathrooms  area   price
0  house1.jpg         3          2  1200  200000
1  house2.jpg         4          3  1800  350000
2  house3.jpg         2          1   900  120000
3  house4.jpg         5          4  2500  500000


## 3. Load Images

In [3]:
IMG_SIZE = 128

def load_images(image_names):
    images = []
    for name in image_names:
        path = os.path.join("data/images", name)
        img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
        img = img_to_array(img) / 255.0
        images.append(img)
    return np.array(images)

X_images = load_images(df["image"])

## 4. Tabular Features

In [4]:
X_tabular = df[["bedrooms", "bathrooms", "area"]].values
y = df["price"].values

## 5. Train Test Split

In [5]:
X_img_train, X_img_test, X_tab_train, X_tab_test, y_train, y_test = train_test_split(
    X_images, X_tabular, y, test_size=0.2, random_state=42
)

## 6. CNN Model for Images

In [6]:
image_input = Input(shape=(128,128,3))

base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_tensor=image_input
)

x = base_model.output
x = Flatten()(x)
x = Dense(128, activation="relu")(x)

image_model = Model(inputs=image_input, outputs=x)

9406464/9406464 [==============================] - 3s 0us/step


## 7. Tabular Model

In [7]:
tabular_input = Input(shape=(3,))

t = Dense(32, activation="relu")(tabular_input)
t = Dense(16, activation="relu")(t)

tabular_model = Model(inputs=tabular_input, outputs=t)

## 8. Feature Fusion (Image + Tabular)

In [8]:
combined = Concatenate()([image_model.output, tabular_model.output])

z = Dense(64, activation="relu")(combined)
z = Dense(1)(z)

model = Model(
    inputs=[image_model.input, tabular_model.input],
    outputs=z
)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 128, 128, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 64, 64, 32)   864         ['input_1[0][0]']                
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 64, 64, 32)   128         ['Conv1[0][0]']                  
                                                                                                  
 Conv1_relu (ReLU)              (None, 64, 64, 32)   0           ['bn_Conv1[0][0]']         

## 9. Train Model

In [9]:
history = model.fit(
    [X_img_train, X_tab_train],
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=8
)

Epoch 1/5
1/1 [==============================] - 10s 10s/step - loss: 145001152512.0000 - mae: 350001.2188 - val_loss: 14396010496.0000 - val_mae: 119983.3750
Epoch 2/5
1/1 [==============================] - 0s 498ms/step - loss: 144987652096.0000 - mae: 349983.2500 - val_loss: 14390176768.0000 - val_mae: 119959.0625
Epoch 3/5
1/1 [==============================] - 0s 363ms/step - loss: 144962043904.0000 - mae: 349953.6250 - val_loss: 14384722944.0000 - val_mae: 119936.3281
Epoch 4/5
1/1 [==============================] - 0s 344ms/step - loss: 144932487168.0000 - mae: 349920.8125 - val_loss: 14379062272.0000 - val_mae: 119912.7266
Epoch 5/5
1/1 [==============================] - 0s 367ms/step - loss: 144903667712.0000 - mae: 349888.7500 - val_loss: 14371801088.0000 - val_mae: 119882.4453


## 10. Evaluate (MAE + RMSE)

In [10]:
pred = model.predict([X_img_test, X_tab_test])

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("MAE:", mae)
print("RMSE:", rmse)

1/1 [==============================] - 1s 1s/step
MAE: 349859.875
RMSE: 349859.8737551936
